In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


df = pd.read_csv('../data/processed/flights_features.csv')
print(df.shape)

(5231130, 41)


In [2]:
feature_cols = [
    'AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 
    'SCHED_DEP_HOUR', 'SCHED_ARR_HOUR', 
    'DEP_HOUR_SIN', 'DEP_HOUR_COS', 'ARR_HOUR_SIN', 'ARR_HOUR_COS',
    'DAY_OF_WEEK', 'IS_WEEKEND', 'MONTH',
    'DISTANCE', 'ROUTE_POPULARITY'
]

X = df[feature_cols]
y = df['IS_DELAY']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (4184904, 14)
Test shape: (1046226, 14)


In [3]:
# Calculate statistic from Train
airline_reliability_train = X_train.join(y_train).groupby('AIRLINE')['IS_DELAY'].mean()

# apply the mapping on test and train both
X_train['AIRLINE_DELAY_RATE'] = X_train['AIRLINE'].map(airline_reliability_train)
X_test['AIRLINE_DELAY_RATE'] = X_train['AIRLINE'].map(airline_reliability_train)

print(X_train[['AIRLINE', 'AIRLINE_DELAY_RATE']].drop_duplicates())
print("\nAny NaN in test?", X_test['AIRLINE_DELAY_RATE'].isnull().sum())

        AIRLINE  AIRLINE_DELAY_RATE
2607226      WN            0.189267
2675694      AA            0.181346
2152178      UA            0.207939
5160633      EV            0.193961
3915376      B6            0.222496
2021522      DL            0.135603
254148       US            0.180398
2502874      MQ            0.219435
232316       F9            0.263979
3716972      OO            0.184452
5165657      AS            0.124737
4092443      HA            0.106898
4588180      NK            0.295573
2243119      VX            0.188260

Any NaN in test? 1046226


In [4]:
print("Train dtype:", X_train['AIRLINE'].dtype)
print("Test dtype:", X_test['AIRLINE'].dtype)
print("\nTrain airline values:", X_train['AIRLINE'].unique()[:5])
print("Test airline values:", X_test['AIRLINE'].unique()[:5])
print("\nMapping keys (index):", airline_reliability_train.index.tolist())
print("\nMismatch check:", set(X_test['AIRLINE'].unique()) - set(airline_reliability_train.index))

Train dtype: str
Test dtype: str

Train airline values: <StringArray>
['WN', 'AA', 'UA', 'EV', 'B6']
Length: 5, dtype: str
Test airline values: <StringArray>
['B6', 'DL', 'UA', 'WN', 'MQ']
Length: 5, dtype: str

Mapping keys (index): ['AA', 'AS', 'B6', 'DL', 'EV', 'F9', 'HA', 'MQ', 'NK', 'OO', 'UA', 'US', 'VX', 'WN']

Mismatch check: set()


In [5]:
airline_map_dict = airline_reliability_train.to_dict()

X_train['AIRLINE_DELAY_RATE'] = X_train['AIRLINE'].astype(str).map(airline_map_dict)
X_test['AIRLINE_DELAY_RATE'] = X_test['AIRLINE'].astype(str).map(airline_map_dict)

print("NaN in train:", X_train['AIRLINE_DELAY_RATE'].isnull().sum())
print("NaN in test:", X_test['AIRLINE_DELAY_RATE'].isnull().sum())

NaN in train: 0
NaN in test: 0


In [6]:
from sklearn.preprocessing import LabelEncoder

X_train_final = X_train.drop(columns=['AIRLINE']).copy()
X_test_final = X_test.drop(columns=['AIRLINE']).copy()


for col in ['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']:
    le = LabelEncoder()
    X_train_final[col] = le.fit_transform(X_train_final[col].astype(str))
    X_test_final[col] = X_test_final[col].astype(str).map(
        dict(zip(le.classes_, le.transform(le.classes_)))
    ).fillna(-1)

print(X_train_final.dtypes)

ORIGIN_AIRPORT           int64
DESTINATION_AIRPORT      int64
SCHED_DEP_HOUR           int64
SCHED_ARR_HOUR           int64
DEP_HOUR_SIN           float64
DEP_HOUR_COS           float64
ARR_HOUR_SIN           float64
ARR_HOUR_COS           float64
DAY_OF_WEEK              int64
IS_WEEKEND               int64
MONTH                    int64
DISTANCE                 int64
ROUTE_POPULARITY         int64
AIRLINE_DELAY_RATE     float64
dtype: object


In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final)
X_test_scaled = scaler.transform(X_test_final)

model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("AUC-ROC:", roc_auc_score(y_test, y_proba))

Accuracy: 0.5704589639332228
Precision: 0.24678269844062944
Recall: 0.6458820607289963
F1 Score: 0.35711618764779346
AUC-ROC: 0.633105916474116


In [10]:
import joblib
joblib.dump(scaler, '../src/scaler_baseline.pkl')

['../src/scaler_baseline.pkl']